# Tema 1 — Lectura y manipulación de datos con pandas

| Recurso | Fichero |
| --- | --- |
| **Ejercicio pandas (E1)** | [`5_ejercicios_pandas.md`](5_ejercicios_pandas.md) |
| Ejercicios (datos + ML) | [`7_ejercicios_datos_y_modelos.md`](7_ejercicios_datos_y_modelos.md) |
| Dataset ejercicio | [`Datos/ventas.csv`](Datos/ventas.csv) |
| Dataset auxiliar | [`Datos/iris.csv`](Datos/iris.csv) |
| Mismos datos iris (JSON / Excel) | [`Datos/iris.json`](Datos/iris.json) · [`Datos/iris.xlsx`](Datos/iris.xlsx) |
| Documentación | [pandas.pydata.org/docs](https://pandas.pydata.org/docs/) |

Este notebook es la **referencia amplia** de la sesión de lectura de datos en DAVD: conceptos, demos ejecutables, flujos recomendados, anti-patrones y puente a modelos y visualizaciones. Úsalo en clase y como manual cuando prepares datos para un cuadro de mando.

**Cómo trabajarlo:** ejecuta las celdas en orden con el entorno del curso activado (`pandas` instalado; para Excel suele hacer falta `openpyxl`).


---

## 0. Objetivos de aprendizaje

Al terminar esta sesión (y este notebook) deberías ser capaz de:

1. Explicar por qué pandas es la base de **lectura y transformación** antes de visualizar o modelar.
2. Distinguir `Series` y `DataFrame` y crearlos de forma controlada.
3. Cargar **CSV, JSON y Excel** con **`pathlib`** (sin rutas absolutas del autor).
4. Diagnosticar un dataset: `shape`, `dtypes`, nulos, distribuciones.
5. Filtrar, seleccionar (`loc`/`iloc`), agrupar y agregar.
6. Crear columnas derivadas con operaciones vectoriales (evitar `apply` innecesario).
7. Exportar resultados listos para la siguiente etapa (modelo o dashboard).
8. Enlazar este flujo con los ejercicios de lectura multi-formato y con scikit-learn.
9. Completar el **E1 pandas** (`5_ejercicios_pandas.md`): diagnosticar, validar, agregar y exportar `ventas.csv`.

---


## 1. Por qué esta sesión importa en DAVD

### 1.1 Qué entregamos

En DAVD no entregamos “un Excel retocado a mano”. Entregamos **aplicaciones y cuadros de mando** que consumen datos de forma reproducible:

```text
datos crudos → lectura → diagnóstico → transformación → (modelo) → visualización / Dash → despliegue
```

Si la lectura y la limpieza viven solo en celdas frágiles, el dashboard se rompe en cuanto cambias de máquina o de fichero.

### 1.2 Mapa mental del semestre

```text
Tema 1  →  Python, pandas (hoy), modelos, viz básicas
Tema 2  →  Plotly / Dash / callbacks
Tema 3  →  entornos, APIs, despliegue
```

Todo lo posterior **asume** que sabes cargar datos de forma portable y dejar un DataFrame limpio.

### 1.3 Analogías útiles

| Concepto | Analogía |
| --- | --- |
| `Series` | Una columna etiquetada |
| `DataFrame` | Tabla (colecciones de Series alineadas) |
| `dtype` | El “tipo de casilla” de la hoja |
| filtro booleano | Quedarse con las filas que cumplen una regla |
| `groupby` | Tablas dinámicas de Excel, pero en código |
| `pathlib` | GPS relativo al proyecto, no “en mi Escritorio” |
| exportar CSV/JSON | Entregar el dataset listo al siguiente módulo (modelo / Dash) |

### 1.4 Regla de oro

> **Explora en el notebook; deja rutas portables; exporta un artefacto que otra persona pueda volver a generar.**

---


## 2. Entorno e imports

Comprueba que el intérprete del notebook es el del entorno del curso.


In [ ]:
from __future__ import annotations

from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd

print("pandas:", pd.__version__)
print("cwd:", Path.cwd())
DATA_DIR = Path("Datos")  # ejecuta desde 1_Introduccion_Python/
print("Datos existe:", DATA_DIR.exists())
print("ficheros:", sorted(p.name for p in DATA_DIR.glob("iris.*")))


Si `Datos existe: False`, cambia el directorio de trabajo del notebook a `1_Introduccion_Python/` (o ajusta `DATA_DIR`).

Para Excel:

```bash
python -m pip install openpyxl
```

---

## 3. `Series`: la columna tipada

Una **Series** es una estructura **unidimensional** con índice. Documentación: [`pandas.Series`](https://pandas.pydata.org/docs/reference/api/pandas.Series.html).

### 3.1 Creación básica


In [ ]:
serie = pd.Series(["a", "b", "c"])
serie


### 3.2 Series con índice semántico (p. ej. fechas)


In [ ]:
ts = pd.Series(
    {
        date(2026, 9, 14): 10,
        date(2026, 9, 21): 15,
        date(2026, 9, 28): 15,
    }
)
ts


**Para DAVD:** casi siempre trabajarás con columnas dentro de un `DataFrame` que luego alimentarás a Plotly/Dash o a un modelo.

---

## 4. `DataFrame`: la tabla de trabajo

Un **DataFrame** es una estructura **bidimensional** (filas × columnas). Cada columna es una Series. Documentación: [`pandas.DataFrame`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html).

### 4.1 Crear desde filas + nombres de columnas


In [ ]:
df_demo = pd.DataFrame(
    data=[
        [date(2026, 9, 14), 1, "a"],
        [date(2026, 9, 21), 2, "b"],
        [date(2026, 9, 28), 3, "c"],
    ],
    columns=["timestamp", "integers", "strings"],
)
df_demo


### 4.2 Índice: cuidado con `inplace`

`set_index` **devuelve una copia** salvo que uses `inplace=True` (preferible asignar).


In [ ]:
df_indexed = df_demo.set_index("timestamp")
df_indexed


### 4.3 Crear columna a columna


In [ ]:
df2 = pd.DataFrame()
df2["timestamp"] = [date(2026, 9, 14), date(2026, 9, 21), date(2026, 9, 28)]
df2["integers"] = [1, 2, 3]
df2["strings"] = ["a", "b", "c"]
df2


In [ ]:
# Comparación elemento a elemento
df_demo.reset_index(drop=True) == df2


### 4.4 Paso a NumPy

Útil cuando pasas features a scikit-learn. Preferible `to_numpy()` frente a `.values`.


In [ ]:
df_demo.to_numpy()


---

## 5. Lectura de datos (I/O en el borde)

### 5.1 Rutas: absoluta vs relativa

| Tipo | Ejemplo | Problema |
| --- | --- | --- |
| Absoluta | `/Users/yo/Desktop/iris.csv` | No funciona en otra máquina |
| Relativa al tema | `Datos/iris.csv` | Portable si fijas el cwd |

En DAVD usamos **`pathlib.Path`** relativo al proyecto.

### 5.2 CSV

Documentación: [`read_csv`](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html).


In [ ]:
iris_csv = DATA_DIR / "iris.csv"
iris = pd.read_csv(iris_csv)
iris.head()


### 5.3 JSON

Documentación: [`read_json`](https://pandas.pydata.org/docs/reference/api/pandas.read_json.html).


In [ ]:
iris_json = pd.read_json(DATA_DIR / "iris.json")
iris_json.head()


### 5.4 Excel

Documentación: [`read_excel`](https://pandas.pydata.org/docs/reference/api/pandas.read_excel.html).


In [ ]:
iris_xlsx = pd.read_excel(DATA_DIR / "iris.xlsx", engine="openpyxl")
iris_xlsx.head()


### 5.5 ¿Son el mismo dataset?

En un dashboard o un modelo, conviene comprobar que el origen no cambia el contenido.


In [ ]:
# Normalizamos nombres de columnas para comparar
def normalize_columns(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    out.columns = [str(c).replace(".", "_") for c in out.columns]
    return out.reset_index(drop=True)


a = normalize_columns(iris)
b = normalize_columns(iris_json)
c = normalize_columns(iris_xlsx)
print("CSV vs JSON iguales:", a.equals(b))
print("CSV vs Excel iguales:", a.equals(c))
print("shape:", a.shape)


### 5.6 Función única de carga (adelanto del ejercicio E1)

Idea que reutilizarás con `airports` u otros ficheros:


In [ ]:
def leer_tabla(path: Path) -> pd.DataFrame:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"No existe el fichero: {path}")
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix == ".json":
        return pd.read_json(path)
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path, engine="openpyxl")
    raise ValueError(f"Formato no soportado: {suffix}")


leer_tabla(DATA_DIR / "iris.csv").head(2)


---

## 6. Diagnóstico de un DataFrame (antes de visualizar o modelar)

Orden recomendado:

```text
cargar → shape/dtypes → head → na → describe/value_counts → sospechosos
```

Trabajamos ya con nombres de columna normalizados.


In [ ]:
iris = normalize_columns(pd.read_csv(DATA_DIR / "iris.csv"))
print("shape:", iris.shape)
print()
print("dtypes:")
print(iris.dtypes)
print()
print("nulos por columna:")
print(iris.isna().sum())


In [ ]:
iris.head()


In [ ]:
iris.describe(include="all")


In [ ]:
iris["variety"].value_counts()


**Lectura crítica:** ¿hay desbalance entre clases? ¿escalas muy distintas entre `sepal_*` y `petal_*`? Eso importa después en ML y en ejes de gráficos.

---

## 7. Selección, filtros y localización

### 7.1 Filtro booleano


In [ ]:
iris[iris["sepal_length"] >= iris["sepal_length"].mean()].head()


### 7.2 `loc` (etiquetas) vs `iloc` (posición)


In [ ]:
iris.loc[0, "sepal_length"]


In [ ]:
iris.iloc[19]


**Heurística:** usa nombres de columna en pipelines y callbacks; `iloc` solo cuando la posición importa de verdad.

---

## 8. Agrupaciones y agregaciones

Útiles para KPIs del dashboard (medias por categoría, máximos, conteos).

### 8.1 Media por categoría


In [ ]:
iris.groupby("variety")["sepal_length"].mean()


### 8.2 Varias métricas con `agg`


In [ ]:
iris.groupby("variety").agg(
    petal_length_median=("petal_length", "median"),
    petal_width_max=("petal_width", "max"),
    n=("variety", "size"),
)


---

## 9. Operaciones vectoriales (preferibles a bucles)

### 9.1 Aritmética columna a columna


In [ ]:
(iris["sepal_length"] / iris["petal_length"]).head()


### 9.2 `map` / `apply`: úsalos con criterio

- **Vectorizado** → rápido y claro.
- `Series.map` → transformaciones elemento a elemento simples.
- `DataFrame.apply(..., axis=1)` → flexible pero lento; último recurso.


In [ ]:
iris["petal_width"].map(lambda x: int(x) if x > 1.5 else x).head()


In [ ]:
# apply por filas (pedagógico)
iris.apply(lambda row: 5 * row["petal_length"] + 6 * row["petal_width"], axis=1).head()


Equivalente vectorizado (preferible):


In [ ]:
(5 * iris["petal_length"] + 6 * iris["petal_width"]).head()


---

## 10. Limpieza básica: tipos, duplicados, columnas

### 10.1 Cambiar tipo


In [ ]:
iris["sepal_width"].astype(str).iloc[0]


### 10.2 Duplicados y borrado de columnas (sin romper el original)


In [ ]:
print("duplicados:", int(iris.duplicated().sum()))
iris_no_dupes = iris.drop_duplicates()
iris_without_width = iris.drop(columns=["sepal_width"])
iris_without_width.head()


**Hábitos:**

- Preferir `drop(columns=[...])`.
- Evitar `inplace=True` en material de curso.
- No mutar el DataFrame “crudo” si luego quieres comparar orígenes.

---

## 11. Mini-pipeline DAVD: de fichero a artefacto listo

Patrón que reutilizarás hacia modelos y Dash:

```text
cargar → normalizar → diagnosticar → transformar → exportar
```

### 11.1 Preparar features y target (puente a scikit-learn)


In [ ]:
def preparar_iris(path: Path) -> tuple[pd.DataFrame, pd.Series]:
    frame = normalize_columns(leer_tabla(path))
    features = frame.drop(columns=["variety"])
    target = frame["variety"]
    return features, target


X, y = preparar_iris(DATA_DIR / "iris.csv")
print("X:", X.shape, "| y clases:", y.value_counts().to_dict())
X.head()


### 11.2 Resumen para un cuadro de mando


In [ ]:
resumen = (
    iris.groupby("variety", as_index=False)
    .agg(
        sepal_length_mean=("sepal_length", "mean"),
        petal_length_mean=("petal_length", "mean"),
        n=("variety", "size"),
    )
    .sort_values("n", ascending=False)
)
resumen


### 11.3 Exportar artefactos


In [ ]:
salida_limpia = DATA_DIR / "iris_limpia.csv"
salida_resumen = DATA_DIR / "iris_resumen.csv"

iris.to_csv(salida_limpia, index=False)
resumen.to_csv(salida_resumen, index=False)
print("escrito:", salida_limpia)
print("escrito:", salida_resumen)


> **Nota:** los CSV generados son salidas. Puedes no versionarlos; sí debes saber regenerarlos.

---

## 12. Anti-patrones frecuentes (lista negra)

1. Rutas absolutas (`/Users/.../iris.csv`).
2. Tres celdas distintas para CSV/JSON/Excel sin una función común.
3. Sobrescribir el DataFrame crudo sin `copy()`.
4. `for row in df.iterrows()` para todo.
5. `apply(axis=1)` cuando basta una operación vectorial.
6. Visualizar o entrenar **antes** de mirar nulos y dtypes.
7. Encadenar 30 transformaciones en una sola celda sin nombres.
8. Depender del orden mágico de “Run All”.
9. Dejar el resultado solo en pantalla: sin CSV/JSON no hay pipeline ni Dash.
10. Mezclar lectura, modelo y layout de Dash en un único bloque ilegible.

---

## 13. De notebook a aplicación (adelanto Tema 2–3)

Hoy exploramos aquí. Más adelante querrás:

```text
datos/          ← CSV/JSON o descarga
src/data.py     ← leer_tabla / preparar_*
src/model.py    ← entrenamiento (notebook 6)
app.py          ← Dash que lee el DataFrame limpio
```

Misma lógica; mejor frontera.

---

## 14. Autoevaluación

1. ¿Qué diferencia una Series de un DataFrame?
2. ¿Por qué `pathlib` frente a rutas absolutas?
3. ¿Cómo cargarías CSV, JSON o Excel con una sola función?
4. ¿Qué imprime tu diagnóstico antes de plotear o entrenar?
5. ¿Cuándo evitarías `apply(axis=1)`?
6. ¿Qué exportarías para alimentar un callback de Dash?

Si dudas en más de dos, rehaz las secciones 5–11 y el ejercicio E1 de [`7_ejercicios_datos_y_modelos.md`](7_ejercicios_datos_y_modelos.md).

---

## 15. Checklist de salida

### Código

- [ ] Notebook ejecuta desde `1_Introduccion_Python/`
- [ ] `iris.csv` / `.json` / `.xlsx` cargados con `Path`
- [ ] Diagnóstico: `shape`, `dtypes`, nulos, `value_counts`
- [ ] Filtros, `groupby` y operación vectorizada probados
- [ ] Función `leer_tabla` (o equivalente) escrita
- [ ] Exportados `iris_limpia.csv` y `iris_resumen.csv`
- [ ] E1 pandas: `validar_ventas` → 140/10 y JSON de calidad

### Comprensión

- [ ] Sé explicar el flujo cargar → transformar → exportar → viz/modelo
- [ ] Sé 3 anti-patrones que no cometeré en el proyecto
- [ ] Sé qué llevaré al notebook de modelos y a Dash

---

## 16. Para la siguiente sesión

1. Completa el **E1 pandas** en [`5_ejercicios_pandas.md`](5_ejercicios_pandas.md) (ventas: 140 válidas / 10 inválidas).
2. Completa el E1 de airports multi-formato en [`7_ejercicios_datos_y_modelos.md`](7_ejercicios_datos_y_modelos.md).
3. Ojea `6_modelos_machine_learning.ipynb`: ahí `X` / `y` ya no serán un misterio.
4. Piensa qué columnas de **tu proyecto** necesitarás leer y limpiar para el dashboard.

---

## 17. Apéndice A — Chuleta rápida

```python
from pathlib import Path
import pandas as pd

DATA = Path("Datos")
df = pd.read_csv(DATA / "iris.csv")
df = df.rename(columns=lambda c: str(c).replace(".", "_"))
df.shape, df.dtypes, df.isna().sum()
df.head(); df.describe(include="all")
df[df["variety"] == "Setosa"]
df.loc[0, "sepal_length"]; df.iloc[0]
df.groupby("variety")["sepal_length"].mean()
df.to_csv(DATA / "salida.csv", index=False)
pd.read_json(DATA / "iris.json")
pd.read_excel(DATA / "iris.xlsx")
```

---

## 18. Apéndice B — Glosario corto EN/ES

| EN | ES / nota |
| --- | --- |
| DataFrame | tabla / marco de datos |
| Series | serie / columna etiquetada |
| dtype | tipo de dato de columna |
| missing / NA | valor ausente / nulo |
| filter / mask | filtro / máscara booleana |
| groupby | agrupación |
| aggregate (`agg`) | agregar / resumir |
| vectorized | vectorizado |
| feature / target | variable predictora / objetivo |
| dashboard | cuadro de mando |

---

## 19. Apéndice C — Preguntas típicas de clase

**¿Puedo hacer la limpieza solo en Excel y pegar el CSV?**  
Para aprender, no: el valor está en el **código reproducible** que luego usará Dash.

**¿Obligatorio usar `pathlib`?**  
Sí en el material del curso: evita rutas de tu portátil.

**¿Por qué falló `read_excel`?**  
Casi seguro falta `openpyxl` (o el motor que uses) en el entorno.

**¿CSV, JSON y Excel deben coincidir siempre?**  
En este material de iris, sí. En la vida real, documenta el formato canónico.

**¿Esto sustituye al notebook de modelos?**  
No. Aquí preparas datos; en `6_modelos_machine_learning.ipynb` entrenas.

---

## 20. Cierre

Si dominas este notebook, tienes la base de datos de DAVD:

> **Leer con rutas portables → diagnosticar → transformar → exportar, listo para modelo o dashboard.**

**Siguiente paso inmediato:** [`5_ejercicios_pandas.md`](5_ejercicios_pandas.md) (ventas) y, si puedes, el E1 de airports en [`7_ejercicios_datos_y_modelos.md`](7_ejercicios_datos_y_modelos.md).


---

## 21. Ejercicio guiado E1 — `ventas.csv` (30 min)

Enunciado completo: [`5_ejercicios_pandas.md`](5_ejercicios_pandas.md).

Objetivo: dejar datos listos para un **cuadro de mando** (válidos vs errores + KPIs + JSON de calidad).

**Checkpoint:** **140 válidas** y **10 inválidas**.

### 21.1 Carga y diagnóstico


In [ ]:
from pathlib import Path
import json
import pandas as pd

DATA_DIR = Path("Datos")
ventas = pd.read_csv(DATA_DIR / "ventas.csv")

print("shape:", ventas.shape)
print(ventas.dtypes)
print(ventas.isna().sum())
ventas.head()


### 21.2 Validación

Implementa `validar_ventas` y comprueba el checkpoint.


In [ ]:
def validar_ventas(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    work = frame.copy()
    work["unidades"] = pd.to_numeric(work["unidades"], errors="coerce")
    work["precio_unitario"] = pd.to_numeric(work["precio_unitario"], errors="coerce")
    ok = (
        work["unidades"].notna()
        & (work["unidades"] > 0)
        & work["precio_unitario"].notna()
        & (work["precio_unitario"] > 0)
    )
    validos = work.loc[ok].copy()
    errores = work.loc[~ok].copy()
    validos["importe"] = validos["unidades"] * validos["precio_unitario"]
    return validos, errores


validos, errores = validar_ventas(ventas)
print(f"válidas: {len(validos)} | inválidas: {len(errores)}")
errores


### 21.3 Agregaciones (KPIs)


In [ ]:
importe_por_region = (
    validos.groupby("region", as_index=False)["importe"]
    .sum()
    .sort_values("importe", ascending=False)
)
top_productos = (
    validos.groupby("producto", as_index=False)["importe"]
    .sum()
    .sort_values("importe", ascending=False)
    .head(3)
)
compras_por_cliente = validos["cliente_id"].value_counts()
clientes_recurrentes = compras_por_cliente[compras_por_cliente > 1]

print("Importe por región:")
print(importe_por_region)
print("\nTop productos:")
print(top_productos)
print("\nClientes recurrentes:")
print(clientes_recurrentes)


### 21.4 Exportación


In [ ]:
validos.to_csv(DATA_DIR / "ventas_limpias.csv", index=False)

calidad = {
    "filas_totales": int(len(ventas)),
    "filas_validas": int(len(validos)),
    "filas_invalidas": int(len(errores)),
    "importe_total": float(validos["importe"].sum()),
}
(DATA_DIR / "calidad_datos.json").write_text(
    json.dumps(calidad, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
calidad


Si el checkpoint no cuadra, revisa las reglas `> 0` y `to_numeric(..., errors="coerce")`. Detalle del enunciado en [`5_ejercicios_pandas.md`](5_ejercicios_pandas.md).
